# Model Training

## What we did so far

### Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest 
from sklearn.neighbors import LocalOutlierFactor 
from sklearn.covariance import EllipticEnvelope  

# Load data
df = pd.read_csv(r"C:\Users\Akshay\Desktop\building-energy-anomaly-detection\data\meters\whole\eda.csv")

# Convert timestamp to datetime and sort
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

### Model (Isolation Forest Model)

In [ ]:
# Using isolation forest model to predict the anomalies in the dataset
# Isolation forest model can make much better predictions with great outcome

# Select only numeric columns for model training (exclude timestamp)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
X = df[numeric_cols].fillna(0)

print(f"Training features: {numeric_cols}")
print(f"Training data shape: {X.shape}")

In [ ]:
# Train Isolation Forest model
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1) 
anomaly_iso = iso_forest.fit_predict(X) 

# Train Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
anomaly_lof = lof.fit_predict(X)

# Train Robust Covariance (Mahalanobis)
robust_cov = EllipticEnvelope(contamination=0.05, random_state=42) 
anomaly_maha = robust_cov.fit_predict(X)  

# Fix: Use addition (+) instead of division for counting votes
# Each model votes: 1 if anomaly (-1), 0 if normal (1)
# Sum all votes and check if at least 2 models agree it is an anomaly
df['anomaly_votes'] = ((anomaly_iso == -1).astype(int) + 
                        (anomaly_lof == -1).astype(int) + 
                        (anomaly_maha == -1).astype(int))

df['is_anomaly'] = (df['anomaly_votes'] >= 2).astype(int)

print("3 models built successfully")
print(f"Anomaly detection complete!")

### Model Evaluation

In [ ]:
# Note: IsolationForest in sklearn does not have feature_importances_
# Alternative: Use decision_function scores to understand anomaly severity
anomaly_scores = iso_forest.decision_function(X)

# Create a dataframe with scores
scores_df = pd.DataFrame({
    'score': anomaly_scores,
    'is_anomaly': df['is_anomaly']
})

# Plot anomaly score distribution
plt.figure(figsize=(10, 6))
plt.hist(anomaly_scores, bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.title("Distribution of Isolation Forest Anomaly Scores")
plt.axvline(x=np.percentile(anomaly_scores, 5), color="red", linestyle="--", 
            label="5th Percentile (threshold)")
plt.legend()
plt.tight_layout()
plt.savefig("anomaly_scores_distribution.png", dpi=300)
plt.show()

In [ ]:
# Feature importance alternative: Use feature means for normal vs anomalous data
feature_importance = {}
for col in numeric_cols:
    normal_mean = df[df["is_anomaly"] == 0][col].mean()
    anomaly_mean = df[df["is_anomaly"] == 1][col].mean()
    # Higher difference indicates more important feature
    feature_importance[col] = abs(anomaly_mean - normal_mean)

# Sort by importance
feature_importance = dict(sorted(feature_importance.items(), 
                                  key=lambda x: x[1], reverse=True))

plt.figure(figsize=(10, 6))
plt.barh(list(feature_importance.keys())[:10], 
         list(feature_importance.values())[:10])
plt.xlabel("Importance (Mean Difference)")  
plt.title("Feature Importance - Normal vs Anomaly")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=300)
plt.show()

In [ ]:
# Anomaly detection results
anomaly_df = df[df["is_anomaly"] == 1] 
print(f"Detected {len(anomaly_df)} anomalies ({len(anomaly_df)/len(df)*100:.1f}%)") 

# Time series visualization of anomalies
plt.figure(figsize=(14, 6)) 

# Convert timestamp to string for plotting if needed
df['timestamp_str'] = df['timestamp'].astype(str)
anomaly_df_plot = df[df["is_anomaly"] == 1]

# Plot normal data
plt.plot(range(len(df)), df["electricity"][:len(df)], 
         label="Normal", alpha=0.7)

# Plot anomalies
if len(anomaly_df_plot) > 0:
    plt.scatter(anomaly_df_plot.index, anomaly_df_plot["electricity"], 
                color="red", label="Anomaly", s=100, zorder=5)

plt.xlabel("Time Index")
plt.ylabel("Electricity")
plt.legend()
plt.title("Electricity Consumption with Detected Anomalies")
plt.tight_layout()
plt.savefig("anomaly_detection_plot.png", dpi=300)
plt.show()

In [ ]:
# Summary statistics
print("\n=== Anomaly Detection Summary ===")
print(f"Total data points: {len(df)}")
print(f"Normal points: {len(df) - len(anomaly_df)}")
print(f"Anomalous points: {len(anomaly_df)}")
print(f"\nAnomaly votes distribution:")
print(df["anomaly_votes"].value_counts().sort_index())

# Save results
df.to_csv("anomaly_detection_results.csv", index=False)
print("\nResults saved to anomaly_detection_results.csv")